In [1]:
import pandas as pd
import re

# Assumes you already have:
infectious_df = pd.read_csv(
    "Dataset/VIOLIN_12-10-2025/interim/infectious_diseases_subset.csv",
    encoding="utf-8-sig",
)
file_reference = "Dataset/VIOLIN_12-10-2025/raw/t_reference_202512101313.csv"

# Load as DataFrames
df_reference  = pd.read_csv(file_reference, encoding="utf-8-sig")

# ---------------------------
# Helpers & normalization
# ---------------------------
def normalize_text(s: str) -> str:
    s = s.replace("\u00A0", " ")  # NBSP
    s = s.replace("’", "'").replace("“", '"').replace("”", '"')
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def is_texty(x) -> bool:
    return isinstance(x, str) and len(x) > 0

TEXT_COLS = infectious_df.select_dtypes(include=["object", "string"]).columns

# Light normalize text columns
for col in TEXT_COLS:
    infectious_df[col] = infectious_df[col].map(lambda x: normalize_text(x) if is_texty(x) else x)

# Patterns
REF_TAG_RE = re.compile(r"\[Ref\s*(\d+):[^\]]+\]")
PMID_TAG_RE = re.compile(r"\[PubMed:\s*([0-9]{5,10})\s*\]", flags=re.I)
PAREN_ETAL_RE = re.compile(r"\(([A-Z][A-Za-z'\-]+)\s+et al\.,\s*(\d{4})\)")

# Counters
def count_pattern(df, pattern):
    total = 0
    for col in TEXT_COLS:
        s = df[col].dropna().astype(str)
        total += s.str.count(pattern).sum()
    return int(total)

def snapshot_counts(df, label):
    counts = {
        "Refs": count_pattern(df, r"\[Ref\s*\d+:"),
        "ParenEtAl": count_pattern(df, r"\([A-Z][A-Za-z'\-]+\s+et al\.,\s*\d{4}\)"),
        "PubMed": count_pattern(df, r"\[PubMed:\s*\d{5,10}\]")
    }
    print(f"\n=== Snapshot: {label} ===")
    for k,v in counts.items():
        print(f"{k}: {v}")
    return counts

# ---------------------------
# Build lookups from df_reference
# ---------------------------
df_ref = df_reference.copy()

# Clean IDs and PMIDs
df_ref["c_reference_id"] = pd.to_numeric(df_ref["c_reference_id"], errors="coerce").astype("Int64")

def pmid_digits_list(x):
    if pd.isna(x): return []
    return re.findall(r"\d{5,10}", str(x))

df_ref["pmids_list"] = df_ref["c_pmid"].apply(pmid_digits_list)

# RefID -> single PMID (only when exactly one)
id_to_single_pmid = (
    df_ref.dropna(subset=["c_reference_id"])
          .set_index("c_reference_id")["pmids_list"]
          .apply(lambda lst: lst[0] if len(lst)==1 else None)
          .to_dict()
)

# Name (normalized) -> single PMID (only when exactly one)
def normalize_ref_name(s: str) -> str:
    s = normalize_text(str(s)).lower()
    if s.endswith("."): s = s[:-1]
    return s

df_ref["name_norm"] = df_ref["c_reference_name"].map(normalize_ref_name)
name_to_pmids = df_ref.groupby("name_norm")["pmids_list"].apply(lambda lists: sorted({p for lst in lists for p in lst}))
name_to_single = name_to_pmids[name_to_pmids.apply(lambda lst: len(lst)==1)].map(lambda lst: lst[0]).to_dict()

# ---------------------------
# Phase B: [Ref###] -> [PubMed: ####] (same as before)
# ---------------------------
def replace_ref_with_pubmed(text):
    if not is_texty(text): 
        return text
    def repl(m):
        rid = int(m.group(1))
        pmid = id_to_single_pmid.get(rid)
        return f"[PubMed: {pmid}]" if pmid else m.group(0)
    return REF_TAG_RE.sub(repl, text)

df_phase_b = infectious_df.copy()
before_b = snapshot_counts(df_phase_b, "BEFORE Phase B")
for col in TEXT_COLS:
    df_phase_b[col] = df_phase_b[col].map(replace_ref_with_pubmed)
after_b = snapshot_counts(df_phase_b, "AFTER Phase B")

# ---------------------------
# Phase A: in-cell mapping for (Lastname et al., YEAR)
# Extract (Lastname et al., YEAR : ... [PubMed: ####]) mappings INSIDE THE SAME CELL
# and replace only those parentheticals in that cell.
# ---------------------------
# Build per-cell mapping function
def build_incell_name_to_pmid(text):
    """
    Returns dict { 'cassataro et al., 2005' : '16177328', ... } for that cell.
    We search for patterns like:
      'Cassataro et al., 2005: ... [PubMed: 16177328]'
    allowing any content between ':' and '[PubMed: ...]'.
    """
    mapping = {}
    if not is_texty(text):
        return mapping
    # Find all PubMed IDs in the cell
    pubmed_spans = [(m.start(), m.end(), m.group(1)) for m in PMID_TAG_RE.finditer(text)]
    if not pubmed_spans:
        return mapping
    # Find all "Name, YEAR:" tokens
    name_year_iter = re.finditer(r"([A-Z][A-Za-z'\-]+)\s+et al\.,\s*(\d{4})\s*:", text)
    # For each name/year, find the nearest PubMed that follows it in the cell
    for ny in name_year_iter:
        start = ny.end()  # after colon
        author_head, year = ny.group(1), ny.group(2)
        key = normalize_ref_name(f"{author_head} et al., {year}")
        # find first PubMed tag that appears AFTER this name/year token
        pmids_after = [pmid for s,e,pmid in pubmed_spans if s >= start]
        if pmids_after:
            # if exactly 1 PMID appears before next name/year or in whole cell, we can be strict:
            # narrow to the region until the next name/year or end of text
            next_ny = next((m for m in name_year_iter if m.start() > ny.start()), None)
            end_limit = next_ny.start() if next_ny else len(text)
            pmids_in_region = [pmid for s,e,pmid in pubmed_spans if start <= s < end_limit]
            chosen = None
            if len(pmids_in_region) == 1:
                chosen = pmids_in_region[0]
            elif len(pmids_after) == 1:  # fallback: only one in the cell after the token
                chosen = pmids_after[0]
            if chosen:
                mapping[key] = chosen
    return mapping

def phase_a_incell_replace(text):
    if not is_texty(text):
        return text
    if "( " not in text and "et al." not in text and "(" not in text:
        return text
    cell_map = build_incell_name_to_pmid(text)
    if not cell_map:
        return text
    def repl(m):
        author_head, year = m.group(1), m.group(2)
        key = normalize_ref_name(f"{author_head} et al., {year}")
        pmid = cell_map.get(key)
        return f"[PubMed: {pmid}]" if pmid else m.group(0)
    return PAREN_ETAL_RE.sub(repl, text)

df_phase_a = df_phase_b.copy()
before_a = snapshot_counts(df_phase_a, "BEFORE Phase A (in-cell)")
for col in TEXT_COLS:
    df_phase_a[col] = df_phase_a[col].map(phase_a_incell_replace)
after_a = snapshot_counts(df_phase_a, "AFTER Phase A (in-cell)")

# ---------------------------
# Phase C: fallback via c_reference_name (only unique names)
# ---------------------------
def replace_parenthetical_with_pubmed_by_table(text):
    if not is_texty(text):
        return text
    def repl(m):
        author_head, year = m.group(1), m.group(2)
        key = normalize_ref_name(f"{author_head} et al., {year}")
        pmid = name_to_single.get(key)
        return f"[PubMed: {pmid}]" if pmid else m.group(0)
    return PAREN_ETAL_RE.sub(repl, text)

df_phase_c = df_phase_a.copy()
before_c = snapshot_counts(df_phase_c, "BEFORE Phase C (fallback)")
for col in TEXT_COLS:
    df_phase_c[col] = df_phase_c[col].map(replace_parenthetical_with_pubmed_by_table)
after_c = snapshot_counts(df_phase_c, "AFTER Phase C (fallback)")

# ---------------------------
# Final result & overall stats
# ---------------------------
print("\n=== Overall delta ===")
def delta(before, after): return {k: after[k]-before[k] for k in before}
overall_before = snapshot_counts(infectious_df, "ORIGINAL (baseline)")
overall_after  = snapshot_counts(df_phase_c, "FINAL (after B + A + C)")

print("Saved CSV: infectious_df_pubmed_final.csv")
df_phase_c.to_csv("Dataset/VIOLIN_12-10-2025/interim/infectious_df_pubmed_final.csv", index=False, encoding="utf-8-sig")



=== Snapshot: BEFORE Phase B ===
Refs: 6740
ParenEtAl: 57875
PubMed: 16761

=== Snapshot: AFTER Phase B ===
Refs: 1579
ParenEtAl: 57854
PubMed: 21922



=== Snapshot: BEFORE Phase A (in-cell) ===
Refs: 1579
ParenEtAl: 57854
PubMed: 21922



=== Snapshot: AFTER Phase A (in-cell) ===
Refs: 1579
ParenEtAl: 27898
PubMed: 51878

=== Snapshot: BEFORE Phase C (fallback) ===
Refs: 1579
ParenEtAl: 27898
PubMed: 51878



=== Snapshot: AFTER Phase C (fallback) ===
Refs: 1579
ParenEtAl: 2635
PubMed: 77141

=== Overall delta ===

=== Snapshot: ORIGINAL (baseline) ===
Refs: 6740
ParenEtAl: 57875
PubMed: 16761



=== Snapshot: FINAL (after B + A + C) ===
Refs: 1579
ParenEtAl: 2635
PubMed: 77141
Saved CSV: infectious_df_pubmed_final.csv


In [2]:
import re
import pandas as pd

infectious_df_pubmed_final = df_phase_c
# 1) Remaining [Ref…]
REF_PATTERN = re.compile(r"\[Ref\s*(\d+):")

remaining_ref_ids = []
for col in infectious_df_pubmed_final.select_dtypes(include=["object","string"]):
    infectious_df_pubmed_final[col].dropna().apply(
        lambda s: remaining_ref_ids.extend(REF_PATTERN.findall(str(s)))
    )

remaining_ref_ids = pd.Series(remaining_ref_ids, dtype="Int64").astype(int)

unique_ref_ids = remaining_ref_ids.nunique()
top_ref_ids = remaining_ref_ids.value_counts().head(20)

print(f"=== Remaining [Ref…] ===")
print(f"Total unreplaced tags : {len(remaining_ref_ids)}")
print(f"Unique RefIDs         : {unique_ref_ids}")
print("\nTop 20 RefIDs still unreplaced:")
print(top_ref_ids)

# 2) Remaining (Author et al., YEAR)
PAREN_PATTERN = re.compile(r"\(([A-Z][A-Za-z'’\-]+)\s+et al\.,\s*(\d{4})\)")

remaining_paren = []
for col in infectious_df_pubmed_final.select_dtypes(include=["object","string"]):
    infectious_df_pubmed_final[col].dropna().apply(
        lambda s: remaining_paren.extend(PAREN_PATTERN.findall(str(s)))
    )

# Normalize to "Lastname et al., YEAR"
remaining_paren = [f"{a} et al., {y}" for a,y in remaining_paren]
remaining_paren = pd.Series(remaining_paren)

unique_paren = remaining_paren.nunique()
top_paren = remaining_paren.value_counts().head(20)

print(f"\n=== Remaining (Author et al., YEAR) ===")
print(f"Total unreplaced tags : {len(remaining_paren)}")
print(f"Unique patterns       : {unique_paren}")
print("\nTop 20 still unreplaced:")
print(top_paren)


=== Remaining [Ref…] ===
Total unreplaced tags : 1579
Unique RefIDs         : 158

Top 20 RefIDs still unreplaced:
1779    112
719      96
200      92
1682     56
1697     50
1690     47
1689     47
1551     46
1409     42
1530     42
732      40
718      36
101      33
1740     33
1461     30
254      30
1527     30
227      28
1777     26
1552     23
Name: count, dtype: int64

=== Remaining (Author et al., YEAR) ===
Total unreplaced tags : 2635
Unique patterns       : 48

Top 20 still unreplaced:
Olsen et al., 2001        275
Delpino et al., 2007      250
Lee et al., 2001          221
Wu et al., 2005           180
Jiang et al., 2007        160
Wang et al., 2009         159
Gu et al., 2003           150
Li et al., 2004           144
Yu et al., 2007           102
Chen et al., 1998          84
Lee et al., 2006           77
Lin et al., 2011           77
Fu et al., 2012            68
Yu et al., 2010            60
O'hagan et al., 2009       56
Yu et al., 2009            55
Price et al., 20

In [3]:
import re
import pandas as pd

# --- Extract remaining (Author et al., YEAR) ---
PAREN_PATTERN = re.compile(r"\(([A-Z][A-Za-z'’\-]+)\s+et al\.,\s*(\d{4})\)")
remaining_paren = []
for col in infectious_df_pubmed_final.select_dtypes(include=["object","string"]):
    infectious_df_pubmed_final[col].dropna().apply(
        lambda s: remaining_paren.extend(PAREN_PATTERN.findall(str(s)))
    )
remaining_paren = [f"{a} et al., {y}" for a,y in remaining_paren]
remaining_paren = pd.Series(remaining_paren)

# --- Build mapping from df_reference ---
df_reference["cite_key"] = df_reference["c_reference_name"].str.strip()
mapping = df_reference.groupby("cite_key")["c_pmid"].apply(lambda x: list(set(x.dropna()))).to_dict()

# --- Classify remaining paren ---
stats = {"ambiguous":0, "unique":0, "no_match":0}
ambig_list = []
for ref in remaining_paren.unique():
    pmids = mapping.get(ref, [])
    if not pmids:
        stats["no_match"] += 1
    elif len(pmids) == 1:
        stats["unique"] += 1
    else:
        stats["ambiguous"] += 1
        ambig_list.append((ref, pmids))

print("=== Ambiguity stats for (Author et al., YEAR) ===")
print(stats)
print("\nExamples of ambiguous mappings:")
for r, pmids in ambig_list[:10]:
    print(f"{r} → PMIDs {pmids}")


=== Ambiguity stats for (Author et al., YEAR) ===
{'ambiguous': 38, 'unique': 0, 'no_match': 10}

Examples of ambiguous mappings:
Al-Mariri et al., 2001 → PMIDs ['11553569', '11447155 ']
Yang et al., 2007 → PMIDs ['17470542', '17239499']
Bielinska et al., 2008 → PMIDs ['18260780', '18057181']
Cassataro et al., 2007 → PMIDs ['17600596', '17428946', '17442465']
Wang et al., 2006 → PMIDs ['16501083', '16820184', '16140431', '16530297 ', '16414158']
Lee et al., 1999 → PMIDs ['10534009', '10024603', '10531231']
Fu et al., 2012 → PMIDs ['22819169', '23000128', '22383953']
Wang et al., 2009 → PMIDs ['19428908', '19733584', '19117681']
Yu et al., 2007 → PMIDs ['17681650', '17570767', '18022294']
Lee et al., 2001 → PMIDs ['11768128', '11500447']


In [4]:
import re
import pandas as pd
import numpy as np

# ---------- 0) Get final DF ----------
try:
    df_final = infectious_df_pubmed_final
except NameError:
    df_final = pd.read_csv("infectious_df_pubmed_final.csv", encoding="utf-8-sig")

# df_reference must be in memory
df_ref = df_reference.copy()

# ---------- 1) Normalization helpers ----------
def normalize_text(s: str) -> str:
    s = str(s)
    s = s.replace("\u00A0", " ").replace("’", "'").replace("“", '"').replace("”", '"')
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_name_key(s: str) -> str:
    s = normalize_text(s).lower()
    if s.endswith("."): s = s[:-1]
    return s

def pmid_list(x):
    if pd.isna(x): return []
    return re.findall(r"\d{5,10}", str(x))

TEXT_COLS = df_final.select_dtypes(include=["object","string"]).columns

# ---------- 2) Build reference lookups ----------
df_ref["c_reference_id"] = pd.to_numeric(df_ref["c_reference_id"], errors="coerce").astype("Int64")
df_ref["pmids_list"] = df_ref["c_pmid"].apply(pmid_list)
df_ref["name_norm"] = df_ref["c_reference_name"].map(normalize_name_key)

# (a) RefID -> pmids_list
id_to_pmids = df_ref.dropna(subset=["c_reference_id"]).set_index("c_reference_id")["pmids_list"]

# (b) Name -> pmids_list
name_to_pmids = (
    df_ref.groupby("name_norm")["pmids_list"]
          .apply(lambda lists: sorted({p for lst in lists for p in lst}))
)

# ---------- 3) Extract remaining [Ref…] after all passes ----------
REF_RE = re.compile(r"\[Ref\s*(\d+):[^\]]+\]")
remaining_ref_ids = []
for col in TEXT_COLS:
    df_final[col].dropna().astype(str).apply(lambda s: remaining_ref_ids.extend(REF_RE.findall(s)))

if len(remaining_ref_ids):
    remaining_ref_ids = pd.Series(remaining_ref_ids, dtype="Int64").astype(int)
    ref_occ = remaining_ref_ids.value_counts().rename_axis("c_reference_id").reset_index(name="occurrences")

    # Join to reference table for reason classification
    ref_join = ref_occ.merge(df_ref[["c_reference_id","pmids_list","c_reference_name","c_title","c_authors","c_year"]],
                             on="c_reference_id", how="left")

    ref_join["in_reference"] = ref_join["c_reference_id"].notna()
    ref_join["pmids_count"] = ref_join["pmids_list"].apply(lambda x: len(x) if isinstance(x, list) else 0)

    ref_join["reason"] = np.select(
        [
            ref_join["in_reference"] == False,
            ref_join["pmids_count"] == 0,
            ref_join["pmids_count"] > 1
        ],
        [
            "missing_in_reference_table",
            "present_but_no_pmid",
            "multiple_pmids_in_reference"   # we skip on purpose
        ],
        default="unexpected_not_replaced_unique_pmid"  # rare: our pipeline missed it
    )

    # Summaries
    ref_reason_unique = ref_join.groupby("reason")["c_reference_id"].nunique().sort_values(ascending=False)
    ref_reason_occ    = ref_join.groupby("reason")["occurrences"].sum().sort_values(ascending=False)
else:
    ref_join = pd.DataFrame(columns=["c_reference_id","occurrences","reason"])
    ref_reason_unique = pd.Series(dtype=int)
    ref_reason_occ    = pd.Series(dtype=int)

# ---------- 4) Extract remaining (Author et al., YEAR) ----------
PAREN_RE = re.compile(r"\(([A-Z][A-Za-z'\-]+)\s+et al\.,\s*(\d{4})\)")
remaining_paren = []
for col in TEXT_COLS:
    df_final[col].dropna().astype(str).apply(lambda s: remaining_paren.extend(PAREN_RE.findall(s)))

if len(remaining_paren):
    paren_keys = [normalize_name_key(f"{a} et al., {y}") for a,y in remaining_paren]
    paren_keys = pd.Series(paren_keys, name="name_norm")
    paren_occ = paren_keys.value_counts().rename_axis("name_norm").reset_index(name="occurrences")

    # Join to name mapping
    name_map = name_to_pmids.reset_index().rename(columns={"pmids_list":"pmids_list_name"})
    paren_join = paren_occ.merge(name_map, on="name_norm", how="left")

    # Classify reasons
    def pmid_len(x): return len(x) if isinstance(x, list) else 0
    paren_join["pmids_count"] = paren_join["pmids_list_name"].apply(pmid_len)
    paren_join["in_reference"] = paren_join["pmids_list_name"].notna()

    paren_join["reason"] = np.select(
        [
            paren_join["in_reference"] == False,
            paren_join["pmids_count"] == 0,
            paren_join["pmids_count"] > 1
        ],
        [
            "no_match_in_reference_name",
            "present_but_no_pmid",
            "ambiguous_multiple_pmids"      # skipped on purpose
        ],
        default="unexpected_not_replaced_unique_pmid"  # rare: should have been replaced by Phase C
    )

    # Summaries
    paren_reason_unique = paren_join.groupby("reason")["name_norm"].nunique().sort_values(ascending=False)
    paren_reason_occ    = paren_join.groupby("reason")["occurrences"].sum().sort_values(ascending=False)
else:
    paren_join = pd.DataFrame(columns=["name_norm","occurrences","reason"])
    paren_reason_unique = pd.Series(dtype=int)
    paren_reason_occ    = pd.Series(dtype=int)

# ---------- 5) Print compact scoreboard ----------
def pretty_block(title, unique_series, occ_series):
    print(f"\n=== {title} ===")
    if unique_series.empty:
        print("Nothing remaining ✅")
        return
    print("By UNIQUE items:")
    for k,v in unique_series.items():
        print(f"  {k:35s} : {v}")
    print("By TOTAL OCCURRENCES:")
    for k,v in occ_series.items():
        print(f"  {k:35s} : {v}")

print("\n######## UNIQUE / OCCURRENCE REASONS ########")
pretty_block("[Ref…] remaining reasons", ref_reason_unique, ref_reason_occ)
pretty_block("(Author et al., YEAR) remaining reasons", paren_reason_unique, paren_reason_occ)

'''# ---------- 6) (Optional) Save detailed CSVs for curation ----------
ref_join.to_csv("unreplaced_refs_by_reason.csv", index=False)
paren_join.to_csv("unreplaced_parentheticals_by_reason.csv", index=False)
print("\nSaved:")
print("  unreplaced_refs_by_reason.csv")
print("  unreplaced_parentheticals_by_reason.csv")
'''


######## UNIQUE / OCCURRENCE REASONS ########

=== [Ref…] remaining reasons ===
By UNIQUE items:
  present_but_no_pmid                 : 158
By TOTAL OCCURRENCES:
  present_but_no_pmid                 : 1579

=== (Author et al., YEAR) remaining reasons ===
By UNIQUE items:
  ambiguous_multiple_pmids            : 38
  no_match_in_reference_name          : 9
  present_but_no_pmid                 : 1
By TOTAL OCCURRENCES:
  ambiguous_multiple_pmids            : 1996
  no_match_in_reference_name          : 364
  present_but_no_pmid                 : 275


'# ---------- 6) (Optional) Save detailed CSVs for curation ----------\nref_join.to_csv("unreplaced_refs_by_reason.csv", index=False)\nparen_join.to_csv("unreplaced_parentheticals_by_reason.csv", index=False)\nprint("\nSaved:")\nprint("  unreplaced_refs_by_reason.csv")\nprint("  unreplaced_parentheticals_by_reason.csv")\n'

In [5]:
import pandas as pd
import re

# ---- source dataframe ----
df = infectious_df_pubmed_final

ID_COL = "c_adjuvant_id"
PMID_RE = re.compile(r"\[PubMed:\s*(\d{5,10})\s*\]")

def extract_pmids_cell(val):
    if isinstance(val, str):
        return sorted(set(PMID_RE.findall(val)))
    return []

# columns to scan
cols_to_scan = [c for c in df.columns if c != ID_COL]

# build stats df (lists in memory)
pmid_stats_df = pd.DataFrame({ID_COL: df[ID_COL]})
for col in cols_to_scan:
    pmid_stats_df[col] = df[col].apply(extract_pmids_cell)

# row-level union + count
pmid_stats_df["row_pmids"] = pmid_stats_df[cols_to_scan].apply(
    lambda r: sorted(set().union(*r)), axis=1
)
pmid_stats_df["row_pmid_count"] = pmid_stats_df["row_pmids"].apply(len)

# ---- drop empty columns safely ----
def col_has_any_pmids(series: pd.Series) -> bool:
    # True if any cell is list/tuple/set with length > 0
    return series.apply(lambda x: isinstance(x, (list, tuple, set)) and len(x) > 0).any()

keep_cols = [ID_COL]
keep_cols += [c for c in cols_to_scan if col_has_any_pmids(pmid_stats_df[c])]
keep_cols += ["row_pmids", "row_pmid_count"]

pmid_stats_df = pmid_stats_df[keep_cols]

# ---- CSV-friendly version (lists -> semicolon-joined strings) ----
pmid_stats_csv = pmid_stats_df.copy()
for col in pmid_stats_csv.columns:
    if col not in (ID_COL, "row_pmid_count"):
        pmid_stats_csv[col] = pmid_stats_csv[col].apply(
            lambda lst: ";".join(lst) if isinstance(lst, (list, tuple, set)) else ""
        )

pmid_stats_csv.to_csv("Dataset/VIOLIN_12-10-2025/interim/infectious_df_pmid_stats_nonempty.csv", index=False, encoding="utf-8-sig")
print("✅ Built stats dataframe and dropped empty columns")
print("✅ Saved as 'infectious_df_pmid_stats_nonempty.csv'")
pmid_stats_csv

✅ Built stats dataframe and dropped empty columns
✅ Saved as 'infectious_df_pmid_stats_nonempty.csv'


,c_adjuvant_id,c_adjuvant_description,c_components,c_function,c_dosage,c_safety,c_preparation,c_description,c_structure,c_appearance,...,c_antigen,c_vector,c_preservative,c_pathogenesis,c_protective_immunity,c_host_range,c_full_text_pathogen,c_introduction,row_pmids,row_pmid_count
0,3,,,4502937,8436836,16043410,6248283,10837643,,,...,,16177328,,12438693;15101970;9060868,12414166,1914022;3089933;9060868;9116636,10482294;10531243;10534009;10816475;10858205;1...,1908158;9060868,10482294;10531243;10534009;10816475;10837643;1...,114
1,4,,,9302736,,,,10930682;9302736,,,...,,,,,,,11254549;12117973;17011680;17351043;18195025;1...,,10930682;11254549;12117973;17011680;17351043;1...,16
2,5,,,9302736,,,,10930682;9302736,,,...,,,,,,,11254549;12117973;17011680;17351043;18195025;1...,,10930682;11254549;12117973;17011680;17351043;1...,16
3,6,,,9302736,,,,10930682;9302736,,,...,,,,,,,11254549;12117973;17011680;17351043;18195025;1...,,10930682;11254549;12117973;17011680;17351043;1...,16
4,7,,,10837642,,,12559801,16795021,7551218,7551218,...,,,,,,,11254549;12117973;17011680;17351043;18195025;1...,,10837642;11254549;12117973;12559801;16795021;1...,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,562,,12649742,12649742,8617961,12649742,,8617961,,,...,12183527,,,20399979,,20399979,10948115;12183554;12798650;18321749;18550728;1...,20399979,10948115;11332721;12183527;12183554;12649742;1...,12
542,563,,,,8436836,3604394,,11739546,,,...,22110377,,,20399979,,20399979,10948115;12183554;12798650;18321749;18550728;1...,20399979,10948115;11739546;12183554;12798650;18321749;1...,12
543,564,,12649742,12649742,8617961,12649742,,8617961,,,...,10618540,,,20399979,,20399979,10948115;12183554;12798650;18321749;18550728;1...,20399979,10618540;10948115;12183554;12649742;12798650;1...,11
544,565,,,4502937,8436836,16043410,6248283,10837643,,,...,10618540,,,20399979,,20399979,10948115;12183554;12798650;18321749;18550728;1...,20399979,10618540;10837643;10948115;12183554;12798650;1...,14


In [6]:
# ---- global unique PMID count ----
all_pmids = set().union(*pmid_stats_df["row_pmids"])
total_unique_pmids = len(all_pmids)

print(f"🌍 Total unique PubMed IDs in dataframe: {total_unique_pmids}")


🌍 Total unique PubMed IDs in dataframe: 1341


In [7]:
# total PMID counts per column (flatten lists)
col_total_counts = {
    col: sum(len(x) for x in pmid_stats_df[col] if isinstance(x, (list, set)))
    for col in pmid_stats_df.columns
    if col not in ("c_adjuvant_id", "row_pmids", "row_pmid_count")
}

# unique PMID counts per column
col_unique_counts = {
    col: len(set().union(*[set(x) for x in pmid_stats_df[col] if isinstance(x, (list, set))]))
    for col in pmid_stats_df.columns
    if col not in ("c_adjuvant_id", "row_pmids", "row_pmid_count")
}

# make summary DataFrame
col_stats = pd.DataFrame({
    "total_pmids": pd.Series(col_total_counts),
    "unique_pmids": pd.Series(col_unique_counts)
}).sort_values("unique_pmids", ascending=False)

print("📊 Column-level PMID stats:")
print(col_stats.head(25))  # top 10


📊 Column-level PMID stats:
                        total_pmids  unique_pmids
c_full_text_pathogen          17535          1211
c_full_text                     421           290
c_adjuvant                      210           136
c_antigen                       190           124
c_adjuvant_description          132            87
c_preparation_vaccine           128            87
c_description_vaccine            97            63
c_protective_immunity           459            52
c_introduction                  588            50
c_pathogenesis                  517            49
c_function                      411            45
c_description                   495            45
c_host_range                    373            40
c_dosage                        248            28
c_safety                        252            25
c_components                    160            22
c_preparation                   248            21
c_virulence                      24            19
c_vector               